In [0]:
from pyspark.sql import functions as f
from pyspark.sql.functions import col, count, when, isnan

catalog_name = 'automobilerepair'

In [0]:
df = spark.read.table("automobilerepair.bronze.stg_invoice")

In [0]:
display(df)

In [0]:
row_count = df.count()
row_count

In [0]:
# DUPLICATE ANALYSIS
print("\nDUPLICATE ANALYSIS")
duplicate_invoice_ids = df.groupBy("invoice_id").count().filter(col("count") > 1)
print(f"Duplicate invoice_ids: {duplicate_invoice_ids.count()}")

# Remove duplicates based on invoice_id, keeping the first occurrence
df = df.dropDuplicates(["invoice_id"])

In [0]:
print("\nNULL VALUE ANALYSIS")

null_counts = df.select([ count(when(col(c).isNull(), c)).alias(c) for c in df.columns ])
print("Null counts by column:")
display(null_counts)

In [0]:
df = df.withColumn("payment_mode", f.when(col("payment_mode").isNull(), "NOT MENTIONED").otherwise(col("payment_mode")))
display(df)

In [0]:
print(df)

In [0]:
#Checking for anamolies handling
df.select("payment_mode").distinct().show()

In [0]:
df.select("currency").distinct().show()

In [0]:
df.write.format("delta") \
    .mode("overwrite") \
    .option("mergeSchema", "true") \
    .saveAsTable(f"{catalog_name}.silver.slv_invoice")